# Модуль 07 · Урок 1. SQL, SQLite и интеграция с pandas

**Как проходить:** изучите все разделы теории последовательно. Запустите демонстрационные ячейки, осознайте входы и выходы методов. Только после этого решайте задания. В условиях используются другие предметные области и комбинации действий — готовых решений в теории нет.

**Обещание урока:** все необходимые операции объяснены ниже; внешний поиск не нужен.

**Стандарт проверки:** правильные значения, форма, порядок и неизменность входа; минимум два набора данных на задачу.

In [ ]:
# Настройка путей для работы из любой папки модуля в VS Code
from pathlib import Path
import os
import sys

ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "tests" / "notebook_checks.py").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError(
        "Не найден корень pandas_academy. Откройте папку курса целиком "
        "через File -> Open Folder в VS Code."
    )

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)  # Теперь data/ и output/ всегда находятся в корне курса

import pandas as pd
import numpy as np
from tests.notebook_checks import expected_for, check_result, show_progress, show_progress

(ROOT / "output").mkdir(exist_ok=True)
print("pandas:", pd.__version__, "| корень проекта:", ROOT)


## 1. Pandas не заменяет СУБД
База данных отвечает за хранение, индексы, ограничения и одновременный доступ. pandas нужен для анализа уже извлечённых данных. Если таблица содержит десятки миллионов строк, не загружайте её полностью только ради двух колонок: фильтруйте и агрегируйте на стороне SQL. `pd.read_sql_query(sql, connection)` возвращает DataFrame. Для учебника достаточно SQLite из стандартной библиотеки Python; при работе с PostgreSQL подход схож, но подключение и транзакции организуются иначе.

In [ ]:
import pandas as pd
import sqlite3
conn = sqlite3.connect(':memory:')
pd.DataFrame({'city':['A','B','A'], 'amount':[2,4,7]}).to_sql('payments',conn,index=False)
print(pd.read_sql_query('SELECT city, SUM(amount) AS total FROM payments GROUP BY city',conn))
conn.close()

## 2. Параметры вместо конкатенации SQL-строк
SQL-инъекции возникают, если внешнее значение вставлять непосредственно в текст запроса. Используйте параметры: `pd.read_sql_query('SELECT ... WHERE status = ?',conn,params=('paid',))` в SQLite. В других драйверах синтаксис placeholder может отличаться. Параметризованным должен быть именно **пользовательский ввод**; имя таблицы или колонки нельзя произвольно передавать параметром, их выбирают из заранее разрешённого списка.

In [ ]:
conn=sqlite3.connect(':memory:')
pd.DataFrame({'code':['X','Y'], 'value':[10,20]}).to_sql('items',conn,index=False)
print(pd.read_sql_query('SELECT value FROM items WHERE code = ?',conn,params=('Y',)))
conn.close()

## 3. Чтение и загрузка
`DataFrame.to_sql(name,conn,if_exists='append',index=False)` добавляет строки, если таблица уже есть. Режим `replace` удаляет существующую таблицу и создаёт новую: в реальной системе это потенциально опасная операция. Для продуктивной загрузки нужно проектировать ключи, транзакции, повторные попытки и проверять, что один и тот же пакет не записался повторно.

## 4. Целостность и границы ответственности
Не все ограничения (FOREIGN KEY, UNIQUE, CHECK) pandas проверяет самостоятельно. Если нужно понять, почему после JOIN выросло число строк, сначала проверьте кардинальность и уникальность ключей. Локальный учебный SQLite не является заменой PostgreSQL в продакшене, но позволяет практиковать переносимость данных и выполнять тесты без серверов.

**Проверьте себя:** когда лучше сделать SUM в SQL, а не в pandas? Почему нельзя собирать SQL с помощью f-строк? Почему `if_exists='replace'` опасен?

---
## Самостоятельная практика

**Инструкция:** выполните ячейку исходных данных, напишите обычный код в следующей ячейке и сохраните итог в `result`. Под решением показан ожидаемый вывод. Затем запустите проверку. **Не нужно писать функции.**

### m7_01 · read sql paid

Прочитайте из SQLite все оплаченные заказы (`status=paid`) с колонками `order_id`,`client_id`,`price`,`qty` в указанном порядке. Сортируйте по order_id.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `db_path` — путь к SQLite-базе, создаётся в ячейке ниже.


**Выполнение:** используйте готовые переменные `db_path`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m7_01: используются переменные db_path

db_path = ROOT / "output" / "examples" / "m7_01_example.db"
db_path.parent.mkdir(parents=True, exist_ok=True)
import sqlite3
db_path_orders = pd.DataFrame({
    'order_id': ['demo_o1', 'demo_o2', 'demo_o3', 'demo_o4', 'demo_o5', 'demo_o6'],
    'client_id': [1, 2, 1, 3, 2, 4],
    'city': ['Москва', 'Тверь', 'Москва', 'Казань', 'Тверь', 'Казань'],
    'category': ['чай', 'кофе', 'чай', 'книги', 'кофе', 'чай'],
    'price': [127.0, 247.0, 97.0, 317.0, 57.0, 147.0],
    'qty': [2, 1, 4, 1, 3, 0],
    'discount': [0, 10, 5, 0, 20, 0],
    'status': ['paid', 'new', 'paid', 'paid', 'cancelled', 'new'],
    'date': ['2026-01-03', '2026-01-04', '2026-01-04', '2026-02-01', '2026-02-02', '2026-02-05'],
    'comment': ['  Быстрая доставка  ', 'без звонка', 'Срочно', 'подарок', '', None],
})
db_path_clients = pd.DataFrame({
    'client_id': [1, 2, 3, 5],
    'segment': ['vip', 'base', 'vip', 'base'],
    'region': ['Центр (демо)', 'Север (демо)', 'Юг (демо)', 'Восток (демо)'],
})
with sqlite3.connect(db_path) as conn:
    db_path_orders.to_sql("orders", conn, if_exists="replace", index=False)
    db_path_clients.to_sql("clients", conn, if_exists="replace", index=False)


# Снимок правильного результата для автопроверки.
_expected_m7_01 = expected_for("m7_01", db_path=db_path)


In [ ]:
# m7_01: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: db_path
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.DataFrame`: 3 строки, 4 столбца; колонки: `order_id`, `client_id`, `price`, `qty`.

```text
  order_id  client_id  price  qty
0  demo_o1          1  127.0    2
1  demo_o3          1   97.0    4
2  demo_o4          3  317.0    1
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m7_01", result, _expected_m7_01)

### m7_02 · read sql minprice

Получите DataFrame SQL-запроса с колонками `order_id`,`price` для price >= min_price, отсортированными по price по убыванию и order_id по возрастанию. Параметризуйте запрос.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `db_path` — путь к SQLite-базе, создаётся в ячейке ниже.
- `min_price` — входной параметр (значение видно в примере); в этом примере: `120`.


**Выполнение:** используйте готовые переменные `db_path`, `min_price`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m7_02: используются переменные db_path, min_price

db_path = ROOT / "output" / "examples" / "m7_02_example.db"
db_path.parent.mkdir(parents=True, exist_ok=True)
import sqlite3
db_path_orders = pd.DataFrame({
    'order_id': ['demo_o1', 'demo_o2', 'demo_o3', 'demo_o4', 'demo_o5', 'demo_o6'],
    'client_id': [1, 2, 1, 3, 2, 4],
    'city': ['Москва', 'Тверь', 'Москва', 'Казань', 'Тверь', 'Казань'],
    'category': ['чай', 'кофе', 'чай', 'книги', 'кофе', 'чай'],
    'price': [127.0, 247.0, 97.0, 317.0, 57.0, 147.0],
    'qty': [2, 1, 4, 1, 3, 0],
    'discount': [0, 10, 5, 0, 20, 0],
    'status': ['paid', 'new', 'paid', 'paid', 'cancelled', 'new'],
    'date': ['2026-01-03', '2026-01-04', '2026-01-04', '2026-02-01', '2026-02-02', '2026-02-05'],
    'comment': ['  Быстрая доставка  ', 'без звонка', 'Срочно', 'подарок', '', None],
})
db_path_clients = pd.DataFrame({
    'client_id': [1, 2, 3, 5],
    'segment': ['vip', 'base', 'vip', 'base'],
    'region': ['Центр (демо)', 'Север (демо)', 'Юг (демо)', 'Восток (демо)'],
})
with sqlite3.connect(db_path) as conn:
    db_path_orders.to_sql("orders", conn, if_exists="replace", index=False)
    db_path_clients.to_sql("clients", conn, if_exists="replace", index=False)

min_price = 120


# Снимок правильного результата для автопроверки.
_expected_m7_02 = expected_for("m7_02", db_path=db_path, min_price=min_price)


In [ ]:
# m7_02: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: db_path, min_price
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.DataFrame`: 4 строки, 2 столбца; колонки: `order_id`, `price`.

```text
  order_id  price
0  demo_o4  317.0
1  demo_o2  247.0
2  demo_o6  147.0
3  demo_o1  127.0
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m7_02", result, _expected_m7_02)

### m7_03 · read sql group

Выполните GROUP BY на стороне SQLite: получите DataFrame `city`,`revenue` (`SUM(price*qty)`), сортировка по city.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `db_path` — путь к SQLite-базе, создаётся в ячейке ниже.


**Выполнение:** используйте готовые переменные `db_path`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m7_03: используются переменные db_path

db_path = ROOT / "output" / "examples" / "m7_03_example.db"
db_path.parent.mkdir(parents=True, exist_ok=True)
import sqlite3
db_path_orders = pd.DataFrame({
    'order_id': ['demo_o1', 'demo_o2', 'demo_o3', 'demo_o4', 'demo_o5', 'demo_o6'],
    'client_id': [1, 2, 1, 3, 2, 4],
    'city': ['Москва', 'Тверь', 'Москва', 'Казань', 'Тверь', 'Казань'],
    'category': ['чай', 'кофе', 'чай', 'книги', 'кофе', 'чай'],
    'price': [127.0, 247.0, 97.0, 317.0, 57.0, 147.0],
    'qty': [2, 1, 4, 1, 3, 0],
    'discount': [0, 10, 5, 0, 20, 0],
    'status': ['paid', 'new', 'paid', 'paid', 'cancelled', 'new'],
    'date': ['2026-01-03', '2026-01-04', '2026-01-04', '2026-02-01', '2026-02-02', '2026-02-05'],
    'comment': ['  Быстрая доставка  ', 'без звонка', 'Срочно', 'подарок', '', None],
})
db_path_clients = pd.DataFrame({
    'client_id': [1, 2, 3, 5],
    'segment': ['vip', 'base', 'vip', 'base'],
    'region': ['Центр (демо)', 'Север (демо)', 'Юг (демо)', 'Восток (демо)'],
})
with sqlite3.connect(db_path) as conn:
    db_path_orders.to_sql("orders", conn, if_exists="replace", index=False)
    db_path_clients.to_sql("clients", conn, if_exists="replace", index=False)


# Снимок правильного результата для автопроверки.
_expected_m7_03 = expected_for("m7_03", db_path=db_path)


In [ ]:
# m7_03: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: db_path
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.DataFrame`: 3 строки, 2 столбца; колонки: `city`, `revenue`.

```text
     city  revenue
0  Казань    317.0
1  Москва    642.0
2   Тверь    418.0
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m7_03", result, _expected_m7_03)

### m7_04 · sql merge report

Прочитайте заказ и сегмент клиента через SQL LEFT JOIN, колонки `order_id`,`client_id`,`segment`, сортируйте по order_id; для неизвестных клиентов segment может быть NULL.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `db_path` — путь к SQLite-базе, создаётся в ячейке ниже.


**Выполнение:** используйте готовые переменные `db_path`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m7_04: используются переменные db_path

db_path = ROOT / "output" / "examples" / "m7_04_example.db"
db_path.parent.mkdir(parents=True, exist_ok=True)
import sqlite3
db_path_orders = pd.DataFrame({
    'order_id': ['demo_o1', 'demo_o2', 'demo_o3', 'demo_o4', 'demo_o5', 'demo_o6'],
    'client_id': [1, 2, 1, 3, 2, 4],
    'city': ['Москва', 'Тверь', 'Москва', 'Казань', 'Тверь', 'Казань'],
    'category': ['чай', 'кофе', 'чай', 'книги', 'кофе', 'чай'],
    'price': [127.0, 247.0, 97.0, 317.0, 57.0, 147.0],
    'qty': [2, 1, 4, 1, 3, 0],
    'discount': [0, 10, 5, 0, 20, 0],
    'status': ['paid', 'new', 'paid', 'paid', 'cancelled', 'new'],
    'date': ['2026-01-03', '2026-01-04', '2026-01-04', '2026-02-01', '2026-02-02', '2026-02-05'],
    'comment': ['  Быстрая доставка  ', 'без звонка', 'Срочно', 'подарок', '', None],
})
db_path_clients = pd.DataFrame({
    'client_id': [1, 2, 3, 5],
    'segment': ['vip', 'base', 'vip', 'base'],
    'region': ['Центр (демо)', 'Север (демо)', 'Юг (демо)', 'Восток (демо)'],
})
with sqlite3.connect(db_path) as conn:
    db_path_orders.to_sql("orders", conn, if_exists="replace", index=False)
    db_path_clients.to_sql("clients", conn, if_exists="replace", index=False)


# Снимок правильного результата для автопроверки.
_expected_m7_04 = expected_for("m7_04", db_path=db_path)


In [ ]:
# m7_04: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: db_path
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.DataFrame`: 6 строк, 3 столбца; колонки: `order_id`, `client_id`, `segment`.

```text
  order_id  client_id segment
0  demo_o1          1     vip
1  demo_o2          2    base
2  demo_o3          1     vip
3  demo_o4          3     vip
4  demo_o5          2    base
5  demo_o6          4    None
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m7_04", result, _expected_m7_04)

### m7_05 · sql top cities

Через SQL посчитайте число заказов в городах и получите `city`,`orders` только для городов с количеством >= min_orders; сортируйте по orders DESC, city ASC.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `db_path` — путь к SQLite-базе, создаётся в ячейке ниже.
- `min_orders` — входной параметр (значение видно в примере); в этом примере: `2`.


**Выполнение:** используйте готовые переменные `db_path`, `min_orders`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m7_05: используются переменные db_path, min_orders

db_path = ROOT / "output" / "examples" / "m7_05_example.db"
db_path.parent.mkdir(parents=True, exist_ok=True)
import sqlite3
db_path_orders = pd.DataFrame({
    'order_id': ['demo_o1', 'demo_o2', 'demo_o3', 'demo_o4', 'demo_o5', 'demo_o6'],
    'client_id': [1, 2, 1, 3, 2, 4],
    'city': ['Москва', 'Тверь', 'Москва', 'Казань', 'Тверь', 'Казань'],
    'category': ['чай', 'кофе', 'чай', 'книги', 'кофе', 'чай'],
    'price': [127.0, 247.0, 97.0, 317.0, 57.0, 147.0],
    'qty': [2, 1, 4, 1, 3, 0],
    'discount': [0, 10, 5, 0, 20, 0],
    'status': ['paid', 'new', 'paid', 'paid', 'cancelled', 'new'],
    'date': ['2026-01-03', '2026-01-04', '2026-01-04', '2026-02-01', '2026-02-02', '2026-02-05'],
    'comment': ['  Быстрая доставка  ', 'без звонка', 'Срочно', 'подарок', '', None],
})
db_path_clients = pd.DataFrame({
    'client_id': [1, 2, 3, 5],
    'segment': ['vip', 'base', 'vip', 'base'],
    'region': ['Центр (демо)', 'Север (демо)', 'Юг (демо)', 'Восток (демо)'],
})
with sqlite3.connect(db_path) as conn:
    db_path_orders.to_sql("orders", conn, if_exists="replace", index=False)
    db_path_clients.to_sql("clients", conn, if_exists="replace", index=False)

min_orders = 2


# Снимок правильного результата для автопроверки.
_expected_m7_05 = expected_for("m7_05", db_path=db_path, min_orders=min_orders)


In [ ]:
# m7_05: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: db_path, min_orders
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.DataFrame`: 3 строки, 2 столбца; колонки: `city`, `orders`.

```text
     city  orders
0  Казань       2
1  Москва       2
2   Тверь       2
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m7_05", result, _expected_m7_05)

---
## Итоги урока

Если все проверки пройдены, повторите своими словами: **что делает каждый метод, какие параметры важны и какая ошибка наиболее вероятна**. Перед переходом к следующему уроку попробуйте решить одно задание заново с чистой ячейки. Автотест подтверждает поведение на известных наборах, но не заменяет код-ревью.

In [ ]:
show_progress()